# Implementação inicial da IA e os testes iniciais.

In [ ]:
from array import array


# Exceções customizadas exigidas pelo professor
class PilhaCheiaErro(Exception):
    """Exceção gerada quando a pilha está cheia."""
    pass


class PilhaVaziaErro(Exception):
    """Exceção gerada quando a pilha está vazia."""
    pass


class TipoErro(Exception):
    """Exceção gerada quando o tipo do dado é inválido."""
    pass


class Pilha:
    def __init__(self, capacidade, tipo):
        if capacidade <= 0:
            raise ValueError("A capacidade deve ser maior que zero.")

        if tipo not in (int, float, str):
            raise TipoErro("O tipo deve ser int, float ou str.")

        self.capacidade = capacidade
        self.tipo = tipo
        self.quantidade = 0

        # Instanciação da classe array com capacidade estática predefinida
        if tipo is int:
            self.dados = array('q', [0] * capacidade)
        elif tipo is float:
            self.dados = array('d', [0.0] * capacidade)
        else:  # tipo is str (caractere)
            self.dados = array('u', ['\0'] * capacidade)

    def empilha(self, dado):
        if self.pilha_esta_cheia():
            raise PilhaCheiaErro("A pilha está cheia.")

        # Validação estrita de tipo
        if self.tipo is str:
            if not isinstance(dado, str) or len(dado) != 1:
                raise TipoErro("O dado deve ser um único caractere.")
        elif type(dado) is not self.tipo:
            raise TipoErro(f"Tipo inválido. Esperado: {self.tipo.__name__}.")

        # Armazena o dado na posição física do topo e incrementa o ponteiro
        self.dados[self.quantidade] = dado
        self.quantidade += 1

    def desempilha(self):
        if self.pilha_esta_vazia():
            raise PilhaVaziaErro("A pilha está vazia.")

        # Decrementa o ponteiro do topo e retorna o elemento
        self.quantidade -= 1
        return self.dados[self.quantidade]

    def pilha_esta_vazia(self):
        return self.quantidade == 0

    def pilha_esta_cheia(self):
        return self.quantidade == self.capacidade

    def troca(self):
        if self.quantidade < 2:
            raise PilhaVaziaErro("É necessário ter pelo menos dois elementos para trocar.")

        topo = self.quantidade - 1
        abaixo = self.quantidade - 2

        # Troca os elementos do topo e do elemento imediatamente abaixo
        self.dados[topo], self.dados[abaixo] = (
            self.dados[abaixo],
            self.dados[topo]
        )

    def tamanho(self):
        return self.quantidade


# --- BATERIA DE TESTES COM SAÍDAS DE IMPRESSÃO ---
if __name__ == "__main__":
    print("==================================================")
    print("   INICIANDO BATERIA DE TESTES DA PILHA (PYTHON)  ")
    print("==================================================\n")

    # 1. Teste de Operações Básicas e Estado (Inteiros)
    print("[TESTE 1] Operações Básicas com Inteiros...")
    p_int = Pilha(3, int)
    assert p_int.pilha_esta_vazia() is True
    print("  ✓ Pilha criada com sucesso e identificada como VAZIA.")

    p_int.empilha(10)
    p_int.empilha(20)
    p_int.empilha(30)
    assert p_int.tamanho() == 3
    assert p_int.pilha_esta_cheia() is True
    print("  ✓ Empilhados 10, 20 e 30. Tamanho = 3. Identificada como CHEIA.")

    removido = p_int.desempilha()
    assert removido == 30
    print(f"  ✓ Desempilhado com sucesso: {removido} (LIFO confirmado).")

    # Teste do Método troca()
    p_int.troca()
    segundo_removido = p_int.desempilha()
    assert segundo_removido == 10
    print(f"  ✓ Método troca() executado. Novo topo desempilhado: {segundo_removido}.")
    print("  [PASS] Teste 1 concluído com sucesso!\n")

    # 2. Cenário de Estresse: Transbordo de Capacidade (PilhaCheiaErro)
    print("[TESTE 2] Cenário de Estresse: Pilha Cheia (Overflow)...")
    p_cheia = Pilha(2, int)
    p_cheia.empilha(100)
    p_cheia.empilha(200)

    try:
        p_cheia.empilha(300)
        assert False, "Erro: Não lançou exceção ao exceder a capacidade!"
    except PilhaCheiaErro as e:
        print(f"  ✓ Exceção PilhaCheiaErro capturada com sucesso: '{e}'")
        print("  [PASS] Teste 2 (Overflow) concluído com sucesso!\n")

    # 3. Cenário de Estresse: Subflutuação (PilhaVaziaErro)
    print("[TESTE 3] Cenário de Estresse: Pilha Vazia (Underflow)...")
    p_vazia = Pilha(3, int)

    try:
        p_vazia.desempilha()
        assert False, "Erro: Não lançou exceção ao desempilhar pilha vazia!"
    except PilhaVaziaErro as e:
        print(f"  ✓ Exceção ao desempilhar capturada com sucesso: '{e}'")

    try:
        p_vazia.troca()
        assert False, "Erro: Não lançou exceção ao trocar com menos de 2 elementos!"
    except PilhaVaziaErro as e:
        print(f"  ✓ Exceção no método troca() capturada com sucesso: '{e}'")
        print("  [PASS] Teste 3 (Underflow) concluído com sucesso!\n")

    # 4. Cenário de Estresse: Validação de Tipo Incompatível (TipoErro)
    print("[TESTE 4] Cenário de Estresse: Tipo Incompatível...")
    p_tipo = Pilha(3, int)

    try:
        p_tipo.empilha(10.5)  # Tenta empilhar float em pilha de int
        assert False, "Erro: Não lançou exceção ao empilhar tipo incompatível!"
    except TipoErro as e:
        print(f"  ✓ Exceção TipoErro capturada com sucesso: '{e}'")
        print("  [PASS] Teste 4 (TipoIncompativel) concluído com sucesso!\n")

    # 5. Teste com Ponto Flutuante (float)
    print("[TESTE 5] Operações com Ponto Flutuante (float)...")
    p_float = Pilha(2, float)
    p_float.empilha(1.5)
    p_float.empilha(2.75)
    assert p_float.desempilha() == 2.75
    print("  ✓ Valores float empilhados e desempilhados corretamente.")
    print("  [PASS] Teste 5 (float) concluído com sucesso!\n")

    # 6. Teste com Caractere (str de tamanho 1)
    print("[TESTE 6] Operações com Caractere (str de tamanho 1)...")
    p_char = Pilha(2, str)
    p_char.empilha('A')
    p_char.empilha('B')
    assert p_char.desempilha() == 'B'
    print("  ✓ Caracteres 'A' e 'B' empilhados e desempilhados com sucesso.")

    try:
        p_char.empilha("TextoLongo")
        assert False, "Erro: Permitiu texto com mais de 1 caractere!"
    except TipoErro as e:
        print(f"  ✓ Rejeição de string longa tratada corretamente: '{e}'")
        print("  [PASS] Teste 6 (char) concluído com sucesso!\n")

    print("==================================================")
    print("   TODOS OS TESTES FORAM EXECUTADOS COM SUCESSO!  ")
    print("==================================================")